# Part 4: AI Race Engineer Chatbot

This notebook builds an **AI Race Engineer chatbot** powered by the **Anthropic Claude API** (model: `claude-sonnet-4-5`) with **tool use**.

The chatbot answers F1 questions by calling Python functions that query real historical data:
- `race_results.csv` — 70+ years of race-by-race results (1950–2024)
- `driver_champions.csv` — annual World Drivers' Championship records

Claude decides which tool(s) to call, executes them against the local CSVs, then synthesises a concise, data-backed answer — just like a real Race Engineer reading from the timing wall.

## 1. Install & Imports

In [ ]:
# pip install anthropic   ← run this if the package isn't already in your venv
# The anthropic package should already be installed; uncomment only if needed.

import anthropic
import pandas as pd
import json
import os

print("anthropic version:", anthropic.__version__)
print("pandas  version:", pd.__version__)

## 2. Load Data

In [ ]:
DATA_DIR = "../data"

# Race results: year, round, race_name, circuit, country, driver,
#               nationality, constructor, grid, laps, decade
df_results = pd.read_csv(f"{DATA_DIR}/race_results.csv")

# Driver champions: year, driver, nationality, constructor,
#                   wins, points, decade
df_champions = pd.read_csv(f"{DATA_DIR}/driver_champions.csv")

print("race_results shape   :", df_results.shape)
print("driver_champions shape:", df_champions.shape)
print()
print("race_results columns :", list(df_results.columns))
print("driver_champions cols :", list(df_champions.columns))

## 3. Define Tools for Claude

We expose five tools to Claude as a JSON schema list.  
Claude reads these descriptions and decides which tool(s) to call when answering a question.

In [ ]:
TOOLS = [
    {
        "name": "get_driver_stats",
        "description": (
            "Return career statistics for a Formula 1 driver: total race wins, "
            "number of World Championships, nationality, constructors raced for, "
            "and the span of active years. Use a partial, case-insensitive name "
            "to handle nicknames or spelling variants."
        ),
        "input_schema": {
            "type": "object",
            "properties": {
                "driver_name": {
                    "type": "string",
                    "description": "Full or partial driver name, e.g. 'Hamilton', 'Verstappen', 'Senna'.",
                }
            },
            "required": ["driver_name"],
        },
    },
    {
        "name": "get_champion_by_year",
        "description": (
            "Return the World Drivers' Champion and the Constructors' Champion "
            "for a specific Formula 1 season."
        ),
        "input_schema": {
            "type": "object",
            "properties": {
                "year": {
                    "type": "integer",
                    "description": "The F1 season year, e.g. 2021.",
                }
            },
            "required": ["year"],
        },
    },
    {
        "name": "get_constructor_stats",
        "description": (
            "Return statistics for a Formula 1 constructor (team): total race wins, "
            "number of Constructors' Championships, and the span of active years. "
            "Wins are counted from race_results; championships from driver_champions "
            "where the constructor appears as champion."
        ),
        "input_schema": {
            "type": "object",
            "properties": {
                "constructor_name": {
                    "type": "string",
                    "description": "Full or partial constructor name, e.g. 'Ferrari', 'McLaren', 'Red Bull'.",
                }
            },
            "required": ["constructor_name"],
        },
    },
    {
        "name": "search_race_results",
        "description": (
            "Search race results by a free-text query. The query is matched "
            "case-insensitively against race_name, circuit, country, driver, "
            "and constructor columns. Returns the top 5 matching rows."
        ),
        "input_schema": {
            "type": "object",
            "properties": {
                "query": {
                    "type": "string",
                    "description": "Search string, e.g. 'Monaco 2019', 'Verstappen wins', 'Silverstone'.",
                }
            },
            "required": ["query"],
        },
    },
    {
        "name": "get_top_drivers",
        "description": (
            "Return the top N drivers ranked by total race wins across all seasons, "
            "optionally filtered by decade."
        ),
        "input_schema": {
            "type": "object",
            "properties": {
                "top_n": {
                    "type": "integer",
                    "description": "Number of drivers to return (default 10).",
                },
                "decade": {
                    "type": "string",
                    "description": (
                        "Optional decade filter, e.g. '2010s', '1990s'. "
                        "If omitted, all decades are included."
                    ),
                },
            },
            "required": [],
        },
    },
]

print(f"Defined {len(TOOLS)} tools for Claude:")
for t in TOOLS:
    print(f"  • {t['name']}")

## 4. Tool Execution Functions

Each function queries `df_results` or `df_champions` and returns a plain Python dict.  
`execute_tool()` acts as the dispatcher — Claude's tool-use loop calls it with the tool name and inputs.

In [ ]:
def get_driver_stats(driver_name: str) -> dict:
    """Return career stats for a driver (case-insensitive partial match)."""
    mask = df_results["driver"].str.contains(driver_name, case=False, na=False)
    driver_races = df_results[mask]

    if driver_races.empty:
        return {"error": f"No race results found for driver matching '{driver_name}'."}

    # Canonical name: most frequent spelling in the data
    canonical_name = driver_races["driver"].mode().iloc[0]

    # Wins = rows where grid == 1  (position 1 on grid means won ... actually
    # race_results doesn't have a 'position' column explicitly named, so we treat
    # grid == 1 as pole, but to count WINS we look at driver_champions wins column
    # and cross-reference. Safer: count rows marked as win via championship data.
    # Since race_results has no explicit 'position' column, we'll infer wins from
    # driver_champions where driver matches, and sum 'wins'.
    champ_mask = df_champions["driver"].str.contains(driver_name, case=False, na=False)
    champ_rows = df_champions[champ_mask]

    total_wins = int(champ_rows["wins"].sum()) if not champ_rows.empty else "N/A"
    championships = int(len(champ_rows))  # seasons in which they were champion

    nationality = driver_races["nationality"].mode().iloc[0]
    constructors = sorted(driver_races["constructor"].dropna().unique().tolist())
    years = sorted(driver_races["year"].dropna().unique().tolist())
    active_years = f"{int(years[0])}–{int(years[-1])}" if years else "N/A"

    return {
        "driver": canonical_name,
        "nationality": nationality,
        "total_race_wins": total_wins,
        "world_championships": championships,
        "constructors_raced_for": constructors,
        "active_years": active_years,
        "seasons_in_data": len(years),
    }


def get_champion_by_year(year: int) -> dict:
    """Return the Drivers' and Constructors' champion for a given season."""
    row = df_champions[df_champions["year"] == int(year)]

    if row.empty:
        return {"error": f"No championship data found for {year}."}

    row = row.iloc[0]
    return {
        "year": int(year),
        "drivers_champion": row["driver"],
        "nationality": row["nationality"],
        "constructor": row["constructor"],
        "wins": int(row["wins"]),
        "points": float(row["points"]),
    }


def get_constructor_stats(constructor_name: str) -> dict:
    """Return career stats for a constructor (case-insensitive partial match)."""
    mask = df_results["constructor"].str.contains(constructor_name, case=False, na=False)
    team_races = df_results[mask]

    if team_races.empty:
        return {"error": f"No race results found for constructor matching '{constructor_name}'."}

    canonical_name = team_races["constructor"].mode().iloc[0]

    # Count seasons where this constructor's driver was champion
    champ_mask = df_champions["constructor"].str.contains(constructor_name, case=False, na=False)
    champ_seasons = df_champions[champ_mask]
    championships = len(champ_seasons)
    total_champ_wins = int(champ_seasons["wins"].sum()) if not champ_seasons.empty else 0

    years = sorted(team_races["year"].dropna().unique().tolist())
    active_years = f"{int(years[0])}–{int(years[-1])}" if years else "N/A"

    # Drivers who raced for this constructor
    drivers = sorted(team_races["driver"].dropna().unique().tolist())

    return {
        "constructor": canonical_name,
        "race_entries": int(len(team_races)),
        "drivers_championship_seasons": championships,
        "total_wins_in_championship_seasons": total_champ_wins,
        "active_years": active_years,
        "notable_drivers": drivers[:20],  # cap at 20 for brevity
    }


def search_race_results(query: str) -> dict:
    """Full-text search across key columns; returns top 5 matching rows."""
    q = query.lower()
    search_cols = ["race_name", "circuit", "country", "driver", "constructor"]

    mask = pd.Series([False] * len(df_results), index=df_results.index)
    for col in search_cols:
        if col in df_results.columns:
            mask |= df_results[col].astype(str).str.lower().str.contains(q, na=False)

    results = df_results[mask].head(5)

    if results.empty:
        return {"error": f"No race results matched '{query}'."}

    return {
        "query": query,
        "matches_found": int(mask.sum()),
        "top_5_results": results[
            [c for c in ["year", "round", "race_name", "circuit", "country", "driver", "constructor", "grid", "laps"]
             if c in results.columns]
        ].to_dict(orient="records"),
    }


def get_top_drivers(top_n: int = 10, decade: str = None) -> dict:
    """Return top N drivers by total race wins, optionally filtered by decade."""
    source = df_champions.copy()

    if decade:
        source = source[source["decade"].astype(str).str.contains(decade, case=False, na=False)]
        if source.empty:
            return {"error": f"No championship data found for decade '{decade}'."}

    leaderboard = (
        source.groupby("driver")["wins"]
        .sum()
        .reset_index()
        .sort_values("wins", ascending=False)
        .head(top_n)
        .rename(columns={"wins": "total_wins"})
    )

    return {
        "decade_filter": decade or "all time",
        "top_drivers": leaderboard.to_dict(orient="records"),
    }


def execute_tool(tool_name: str, tool_input: dict) -> dict:
    """Dispatch a tool call to the appropriate Python function."""
    dispatch = {
        "get_driver_stats": get_driver_stats,
        "get_champion_by_year": get_champion_by_year,
        "get_constructor_stats": get_constructor_stats,
        "search_race_results": search_race_results,
        "get_top_drivers": get_top_drivers,
    }
    fn = dispatch.get(tool_name)
    if fn is None:
        return {"error": f"Unknown tool: {tool_name}"}
    return fn(**tool_input)


# --- Quick sanity-check (no API call needed) ---
print("Tool sanity checks:")
print(json.dumps(get_champion_by_year(2021), indent=2))
print()
print(json.dumps(get_top_drivers(top_n=5), indent=2))

## 5. Build the Chatbot

`chat()` implements the **tool-use agentic loop**:

1. Send the user message to Claude with the tool list.
2. If Claude responds with `stop_reason == "tool_use"`, execute every requested tool and feed the results back.
3. Repeat until Claude returns a plain text answer (`stop_reason == "end_turn"`).

In [ ]:
# ---------------------------------------------------------------------------
# API client
# Set the ANTHROPIC_API_KEY environment variable, or paste your key below.
# ---------------------------------------------------------------------------
client = anthropic.Anthropic(
    api_key=os.environ.get("ANTHROPIC_API_KEY", "YOUR_KEY_HERE")
)

SYSTEM_PROMPT = """You are an expert F1 Race Engineer AI assistant with access to 70+ years \
of Formula 1 data (1950–2024).
You have tools to query race results, driver stats, constructor stats, and championship history.
When answering questions, use your tools to get accurate data. Be concise, enthusiastic, \
and use F1 terminology.
Always cite specific numbers from the data."""


def chat(user_message: str, conversation_history: list = None):
    """
    Send a user message to Claude and handle tool use in a loop.

    Parameters
    ----------
    user_message : str
        The question or statement from the user.
    conversation_history : list, optional
        Existing conversation context (list of message dicts).
        Pass the list returned by a previous call to continue a conversation.

    Returns
    -------
    (str, list)
        The final text answer from Claude, and the updated conversation history.
    """
    if conversation_history is None:
        conversation_history = []

    # Add the new user turn
    conversation_history.append({"role": "user", "content": user_message})

    while True:
        response = client.messages.create(
            model="claude-sonnet-4-5",
            max_tokens=1024,
            system=SYSTEM_PROMPT,
            tools=TOOLS,
            messages=conversation_history,
        )

        # ------------------------------------------------------------------ #
        # Tool-use branch: Claude wants to call one or more tools             #
        # ------------------------------------------------------------------ #
        if response.stop_reason == "tool_use":
            # 1. Record the assistant turn (includes ToolUse blocks)
            conversation_history.append(
                {"role": "assistant", "content": response.content}
            )

            # 2. Execute every tool Claude asked for
            tool_results = []
            for block in response.content:
                if block.type == "tool_use":
                    print(f"  [tool call] {block.name}({block.input})")
                    result = execute_tool(block.name, block.input)
                    tool_results.append(
                        {
                            "type": "tool_result",
                            "tool_use_id": block.id,
                            "content": json.dumps(result),
                        }
                    )

            # 3. Feed tool results back as a user turn
            conversation_history.append(
                {"role": "user", "content": tool_results}
            )
            # Loop: Claude will now synthesise the results into a final answer

        # ------------------------------------------------------------------ #
        # Final answer: Claude is done calling tools                          #
        # ------------------------------------------------------------------ #
        else:
            final_text = next(
                (b.text for b in response.content if hasattr(b, "text")),
                "(no text response)"
            )
            conversation_history.append(
                {"role": "assistant", "content": final_text}
            )
            return final_text, conversation_history


print("Chatbot ready. 'chat()' function defined.")

## 6. Demo Conversations

Run five example questions in a single shared conversation so Claude can build on prior context.

In [ ]:
questions = [
    "Who has the most F1 race wins of all time?",
    "Tell me about Lewis Hamilton's career stats",
    "Who won the 2021 championship and with which team?",
    "Which constructor has the most race wins?",
    "Who are the top 5 drivers by wins in the 2010s decade?",
]

history = []
for q in questions:
    print(f"\n{'='*60}")
    print(f"YOU: {q}")
    print("-" * 60)
    answer, history = chat(q, history)
    print(f"RACE ENGINEER: {answer}")

print(f"\n{'='*60}")
print("Demo complete.")

## 7. Interactive Mode

Uncomment the cell below to start a live chat session directly in the notebook (or terminal).  
Type `quit`, `exit`, or `q` to end the session.

In [ ]:
# Uncomment the block below to run an interactive chat session.
#
# history = []
# print("F1 Race Engineer Chatbot — type 'quit' to exit\n")
# while True:
#     user_input = input("You: ").strip()
#     if not user_input:
#         continue
#     if user_input.lower() in ("quit", "exit", "q"):
#         print("Chequered flag! Session ended.")
#         break
#     answer, history = chat(user_input, history)
#     print(f"Race Engineer: {answer}\n")

## 8. API Key Setup

### How to set your Anthropic API key

**Option A — Environment variable (recommended)**

```bash
# Linux / macOS / WSL
export ANTHROPIC_API_KEY="sk-ant-..."

# Windows PowerShell
$env:ANTHROPIC_API_KEY = "sk-ant-..."

# Windows Command Prompt
set ANTHROPIC_API_KEY=sk-ant-...
```

Then launch Jupyter **after** setting the variable so the notebook inherits it.

**Option B — Paste directly in the notebook**

In the chatbot cell, replace `"YOUR_KEY_HERE"` with your actual key string.  
Do **not** commit this to version control.

**Get your key** at [console.anthropic.com](https://console.anthropic.com) → API Keys → Create key.

---

*Part 4 complete — you now have a fully functional AI Race Engineer powered by Claude and real F1 data.*